In [ ]:
# scripts/run_vrp_sample.py
import os
from pathlib import Path
import pandas as pd
import json
import logging

# ensure project root is one level up if running from scripts/
ROOT = Path.cwd()
# adjust if necessary
DATA_RAW = ROOT / "data" / "sample"
PROC_DIR = ROOT / "data" / "processed"
MAPS_DIR = PROC_DIR / "maps"
DATA_RAW.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# small helper to write CSVs
def to_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print("Wrote", path)

# ------------- 1) Create sample files -------------
# We'll create one branch "main" and batch "morning".
# School coordinate (a made-up point)
school = {"school_name":"Sample School", "school_branch":"main", "school_lat":19.075983, "school_lon":72.877655}  # Mumbai-ish


In [ ]:
import pandas as pd


In [ ]:
# Parking depot (parking spot)
parking = pd.DataFrame([
    {"parking_spot":"Depot A", "parking_lat":19.070000, "parking_lon":72.870000, "buses":10}
])
parking.to_csv(DATA_RAW / "bus_parking_spot.csv")

# Bus master (simple small fleet)
bus_master = pd.DataFrame([
    {"vendor":"test_co", "seating_capacity":50, "available_buses":5, "fixed_cost":150000, "empty_seats":4},
    {"vendor":"mini_co", "seating_capacity":27, "available_buses":3, "fixed_cost":80000, "empty_seats":3}
])
bus_master.to_csv(DATA_RAW / "bus_master.csv")


In [ ]:
# Student travel time table
stt = pd.DataFrame([
    {"distance(km)":2, "travel_time(mins)":20},
    {"distance(km)":5, "travel_time(mins)":45},
    {"distance(km)":10, "travel_time(mins)":70},
    {"distance(km)":15, "travel_time(mins)":120},
    {"distance(km)":20, "travel_time(mins)":180}
])
stt.to_csv(DATA_RAW / "Student_Travel_Time.csv")

# Create stops (6 stops around the school within a few km)
stops = [
    {"stop_id":"S1", "stop_lat":19.0800, "stop_lon":72.8800, "num_participants":6, "wait_time_min":2, "school_branch":"main", "batch":"morning"},
    {"stop_id":"S2", "stop_lat":19.0730, "stop_lon":72.8805, "num_participants":8, "wait_time_min":3, "school_branch":"main", "batch":"morning"},
    {"stop_id":"S3", "stop_lat":19.0685, "stop_lon":72.8750, "num_participants":4, "wait_time_min":2, "school_branch":"main", "batch":"morning"},
    {"stop_id":"S4", "stop_lat":19.0660, "stop_lon":72.8830, "num_participants":7, "wait_time_min":3, "school_branch":"main", "batch":"morning"},
    {"stop_id":"S5", "stop_lat":19.0725, "stop_lon":72.8700, "num_participants":5, "wait_time_min":2, "school_branch":"main", "batch":"morning"},
    {"stop_id":"S6", "stop_lat":19.0795, "stop_lon":72.8720, "num_participants":3, "wait_time_min":2, "school_branch":"main", "batch":"morning"},
]
stops_df = pd.DataFrame(stops)
stops_df.to_csv(DATA_RAW / "stops_pickup.csv")
# participants_with_stops: each participant row is simplified as one row per stop with counts
parts = []
for s in stops:
    parts.append({"id": f"grp_{s['stop_id']}", "type":"student", "stop_id": s['stop_id'], "school_branch":"main", "batch":"morning", "must_have_seat":False})
parts_df = pd.DataFrame(parts)
parts_df.to_csv(DATA_RAW / "participants_with_stops_pickup.csv")


In [ ]:
cfg = {
    "paths": {
        "stops_pickup": str(DATA_RAW / "stops_pickup.csv"),
        "participants_with_stops_pickup": str(DATA_RAW / "participants_with_stops_pickup.csv"),
        "bus_master": str(DATA_RAW / "bus_master.csv"),
        "parking_spots": str(DATA_RAW / "bus_parking_spot.csv"),
        "student_travel_time": str(DATA_RAW / "Student_Travel_Time.csv"),
        "route_cache": str(ROOT / "data" / "route_cache.csv")
    },
    "constraints": {
        "max_occupancy": 0.9,
        "max_bus_km_per_month": 2500
    },
    "routing": {
        "avg_speed_kmph": 25,
        "use_google_directions": True,   # keep False so RouteFetcher falls back to haversine
        "max_vehicles_per_branch": 6,
        "search_time_limit_seconds": 20,
        "school_days_per_month": 22
    },
    "validation": {"max_city_radius_km": 50.0},
    "geocode": {"sleep_between_calls": 0.1}
}
cfg_path = ROOT / "config" / "config_sample_vrp.yaml"
cfg_path.parent.mkdir(parents=True, exist_ok=True)
with open(cfg_path, "w") as fh:
    import yaml
    yaml.safe_dump(cfg, fh)
print("Wrote config to", cfg_path)


In [ ]:
ROOT

In [ ]:
import sys
import os

notebook_dir = os.getcwd()

project_root = os.path.dirname(notebook_dir)


if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.tripgen.vrp_optimizer import VRPTripOptimizer
from src.routing.route_fetcher import RouteFetcher


# initialize logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("vrp_sample")

In [ ]:
# load files we just wrote
stops_df = pd.read_csv(DATA_RAW / "stops_pickup.csv")
parts_df = pd.read_csv(DATA_RAW / "participants_with_stops_pickup.csv")
bus_master_df = pd.read_csv(DATA_RAW / "bus_master.csv")
parking_df = pd.read_csv(DATA_RAW / "bus_parking_spot.csv")
stt_df = pd.read_csv(DATA_RAW / "Student_Travel_Time.csv")

In [ ]:
stt_df.shape

In [ ]:

# prepare bus_types list for VRP optimizer
bus_types = []
for _, r in bus_master_df.iterrows():
    seating = int(r['seating_capacity'])
    empty = r.get('empty_seats', None)
    if pd.isna(empty) or empty == '':
        usable = int(seating * cfg['constraints']['max_occupancy'])
    else:
        usable = seating - int(empty)
    bus_types.append({
        "vendor": r['vendor'],
        "seating_capacity": seating,
        "available_buses": int(r['available_buses']),
        "usable_capacity": usable,
        "fixed_cost": float(r['fixed_cost'])
    })

In [ ]:
bus_types

In [ ]:
# create VRP optimizer
vrp = VRPTripOptimizer()

In [ ]:
# attach student_travel_time table for allowed travel time lookups if needed
vrp.student_travel_time = stt_df.to_dict("records")


In [ ]:
# define depot & school coords
depot = (float(parking_df.iloc[0]['parking_lat']), float(parking_df.iloc[0]['parking_lon']))
school_coord = (school['school_lat'], school['school_lon'])
# set school_start_time (08:30 -> 510)
school_start_time_min = 8*60 + 30

In [ ]:
# solve for branch main & batch morning (we only have one)
result = vrp.solve_branch(stops_df, parts_df, branch="main", batch="morning",
                          bus_types=bus_types,
                          depot=depot,
                          school_coord=school_coord,
                          school_start_time_min=school_start_time_min,
                          logger=logger)

print("VRP result status:", result.get('status'))

In [ ]:
routes_df = result.get('routes_df', pd.DataFrame())
trip_records = result.get('trip_records', [])
print("Routes generated:", len(routes_df))
if not routes_df.empty:
    print(routes_df)

In [ ]:

# Save routes to CSV
routes_out = PROC_DIR / "vrp_routes_sample.csv"
routes_df.to_csv(routes_out, index=False)
print("Saved VRP routes to", routes_out)


In [ ]:


# Visualize routes with Folium 
import folium
from folium.features import DivIcon

m = folium.Map(location=[school_coord[0], school_coord[1]], zoom_start=14)

# add depot marker
folium.Marker(location=list(depot), popup="Depot", icon=folium.Icon(color="green")).add_to(m)
# add school marker
folium.Marker(location=list(school_coord), popup="School", icon=folium.Icon(color="blue", icon="info-sign")).add_to(m)

# add stops
for _, r in stops_df.iterrows():
    folium.CircleMarker(location=[r['stop_lat'], r['stop_lon']],
                        radius=5,
                        color='orange',
                        fill=True,
                        popup=f"{r['stop_id']} ({int(r['num_participants'])})").add_to(m)

# draw each route as a polyline (connect depot -> stops -> school)
import math
for _, row in routes_df.iterrows():
    stops_list = row['stops'].split(",") if row['stops'] else []
    coords = [depot]
    for sid in stops_list:
        sr = stops_df[stops_df['stop_id'] == sid].iloc[0]
        coords.append((float(sr['stop_lat']), float(sr['stop_lon'])))
    coords.append(school_coord)
    folium.PolyLine(locations=coords, weight=4, color='red', opacity=0.7).add_to(m)
    # label first point with trip id
    if len(coords) >= 2:
        folium.map.Marker(
            coords[1],
            icon=DivIcon(icon_size=(150,36), icon_anchor=(0,0),
                         html=f'<div style="font-size:10pt;color:black">{row["trip_id"]}</div>')
        ).add_to(m)

map_out = MAPS_DIR / "vrp_sample_map.html"
m.save(map_out)
print("Saved map to", map_out)
print("Open that HTML file in a browser to visualize routes.")
